# Fused Model (CNN-LSTM + ELM)

## 1. Import Libraries

In [4]:
from tensorflow.keras.models import load_model
import numpy as np

import joblib
import torch
import os
import librosa

import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

In [5]:
#Loading audio model
audio_model = torch.jit.load("cnn_lstm_audio_model_scripted.pt")
audio_model.eval()
#Loading ELM model
elm_model = joblib.load("elm_model.pkl")

In [6]:
# Must match training shape: (batch_size, channels, mel_bins, time_frames)
# Model was trained on: max_len=500 time frames, 128 mel bins
audio_input = np.zeros((1, 1, 128, 500), dtype=np.float32)

# Convert to torch tensor
audio_input = torch.tensor(audio_input)

# Clinical input must have 10 features (same as training data)
clinical_input = np.zeros((1, 10))    

RuntimeError: Could not infer dtype of numpy.float32

In [ ]:
import pandas as pd

def elm_predict_proba(model_dict, X):
    # Predicts using the ELM model from the saved dictionary
    # Extract model components
    w = model_dict['w']
    beta = model_dict['beta']
    b = model_dict['b']
    scaler = model_dict['scaler']
     
    # Convert to DataFrame with original feature names if X is not already DataFrame
    if not isinstance(X, pd.DataFrame):
        # Get original feature names from scaler
        if hasattr(scaler, 'feature_names_in_'):
            feature_names = scaler.feature_names_in_
            # Convert X to DataFrame
            if isinstance(X, np.ndarray):
                X = pd.DataFrame(X, columns=feature_names)
            elif torch.is_tensor(X):
                X = pd.DataFrame(X.numpy(), columns=feature_names)
        else:
            # If no feature names, just convert to array
            if torch.is_tensor(X):
                X = X.numpy()
    
    # Scale input
    X_scaled = scaler.transform(X)
    
    # Define sigmoid function
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))  # Clip to prevent overflow
    
    # Hidden layer activation
    h = sigmoid(np.dot(X_scaled, w) + b)
    
    # Output prediction (probability)
    y_pred_prob = np.dot(h, beta)
    return y_pred_prob.flatten()[0]

# Audio model prediction
p_audio = audio_model(audio_input)

# Clinical model prediction (extract probability from ELM)
p_clinical = elm_predict_proba(elm_model, clinical_input)

In [ ]:
w_audio = 0.7
w_clinical = 0.3

# Convert audio prediction to scalar probability (class 1)
# p_audio has logits for both classes, convert to probability for class 1
if isinstance(p_audio, torch.Tensor):
    p_audio_probs = torch.softmax(p_audio, dim=1)
    p_audio_scalar = p_audio_probs[0, 1].item()  # Probability of class 1
else:
    p_audio_scalar = p_audio[0]

# Extract scalar from clinical prediction if needed
p_clinical_scalar = p_clinical.item() if isinstance(p_clinical, np.ndarray) else float(p_clinical)

print(f"Audio prediction (class 1 prob): {p_audio_scalar:.4f}")
print(f"Clinical prediction (prob): {p_clinical_scalar:.4f}")

# Weighted fusion
p_final = (w_audio * p_audio_scalar) + (w_clinical * p_clinical_scalar)
print(f"Fused prediction: {p_final:.4f}")

Audio prediction (class 1 prob): 0.5740
Clinical prediction (prob): 0.0000
Fused prediction: 0.4018


In [ ]:
prediction = 1 if p_final >= 0.5 else 0

## 2. Model Evaluation on Test Data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import pandas as pd

# Load clinical test data
df = pd.read_csv('../../clinical_data/neonatal_processed.csv')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

x_clinical = df.drop("primary_outcome", axis=1)
y_clinical = df["primary_outcome"]

# Split the data (use same split as original ELM model)
x_train, x_test, y_train, y_test = train_test_split(x_clinical, y_clinical, test_size=0.25, random_state=42)

print(f"Test set: {len(x_test)} samples | Classes: {y_test.value_counts().to_dict()}")


Test set: 7500 samples | Classes: {0: 7296, 1: 204}


In [ ]:
# Evaluate Clinical Model (ELM)
clinical_preds_proba = []
for i in range(len(x_test)):
    test_sample = x_test.iloc[[i]].values
    pred_prob = elm_predict_proba(elm_model, test_sample)
    clinical_preds_proba.append(pred_prob)

clinical_preds_proba = np.array(clinical_preds_proba)
clinical_preds = (clinical_preds_proba >= 0.5).astype(int)

# Metrics
clinical_acc = accuracy_score(y_test, clinical_preds)
clinical_prec = precision_score(y_test, clinical_preds, zero_division=0)
clinical_rec = recall_score(y_test, clinical_preds, zero_division=0)
clinical_f1 = f1_score(y_test, clinical_preds, zero_division=0)
clinical_auc = roc_auc_score(y_test, clinical_preds_proba)

print(f"Clinical Model (ELM): Acc={clinical_acc:.4f} | Prec={clinical_prec:.4f} | Rec={clinical_rec:.4f} | F1={clinical_f1:.4f} | AUC={clinical_auc:.4f}")

Clinical Model (ELM): Acc=0.9845 | Prec=0.8492 | Rec=0.5245 | F1=0.6485 | AUC=0.9579


In [ ]:
import requests
import os
import io
import numpy as np
import torch
import librosa
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def create_session_with_retries():
    session = requests.Session()
    retry = Retry(connect=3, backoff_factor=0.5)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

# Load GitHub token for rate limiting
try:
    from dotenv import load_dotenv
    load_dotenv()
    github_token = os.getenv('GITHUB_TOKEN')
    if github_token:
        headers = {'Authorization': f"token {github_token}"}
        print("✓ GitHub token loaded")
    else:
        headers = {}
        print("⚠ No GitHub token found - rate limits may apply")
except:
    headers = {}
    print("⚠ dotenv not found - using no authentication")

session = create_session_with_retries()

# Get test audio file URLs
print("\n" + "="*70)
print("FETCHING SPRSOUND TEST DATA")
print("="*70)

test_audio_urls = []
test_labels = {}

# Try different possible paths for SPRSound data
possible_paths = [
    "https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/BioCAS2022/test2022_wav",
    "https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/test_wav",
    "https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/data/test",
]

for api_url in possible_paths:
    try:
        print(f"\nTrying: {api_url}")
        response = session.get(api_url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            test_files = response.json()
            
            # Check if response is a list (directory contents)
            if isinstance(test_files, list):
                test_audio_urls = [f['download_url'] for f in test_files if f['name'].endswith('.wav')]
                print(f"✓ Found {len(test_audio_urls)} audio files")
                break
            else:
                print(f"  Response is not a list (type: {type(test_files)})")
        else:
            print(f"  Status code: {response.status_code}")
    except Exception as e:
        print(f"  Error: {str(e)[:100]}")
        continue

# If no audio files found, use local test files or create dummy data
if len(test_audio_urls) == 0:
    print("\n⚠ No remote audio files found. Using local test files if available...")
    
    # Check for local test data
    local_test_path = 'data/test_audio'
    if os.path.exists(local_test_path):
        test_audio_urls = [os.path.join(local_test_path, f) for f in os.listdir(local_test_path) if f.endswith('.wav')][:50]
        print(f"✓ Found {len(test_audio_urls)} local audio files")
    else:
        print("⚠ No local test data found. Creating dummy test data for demonstration...")
        # Create dummy data for testing
        num_dummy_samples = 20
        for i in range(num_dummy_samples):
            test_audio_urls.append(f"dummy_{i}.wav")
            test_labels[f"dummy_{i}.wav"] = np.random.randint(0, 2)

# Load labels from JSON files
print("\nFetching test labels...")
label_paths = [
    "https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/BioCAS2022/test2022_json/intra_test_json",
    "https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/BioCAS2022/test2022_json/inter_test_json",
]

for api_url in label_paths:
    try:
        response = session.get(api_url, headers=headers, timeout=10)
        if response.status_code == 200:
            label_files = response.json()
            if isinstance(label_files, list):
                for lf in label_files[:25]:
                    if lf['name'].endswith('.json'):
                        try:
                            label_response = session.get(lf['download_url'], headers=headers, timeout=10)
                            if label_response.status_code == 200:
                                label_data = label_response.json()
                                # Extract label (adjust key based on actual JSON structure)
                                label = label_data.get('diagnosis_label', label_data.get('label', label_data.get('class', 0)))
                                test_labels[lf['name'].replace('.json', '.wav')] = label
                        except Exception as e:
                            continue
    except Exception as e:
        continue

print(f"✓ Loaded {len(test_labels)} labels")

# Process audio samples
print("\n" + "="*70)
print("PROCESSING AUDIO SAMPLES")
print("="*70)

audio_test_preds = []
audio_test_preds_proba = []
audio_test_true = []

# Limit to 50 samples
max_samples = min(50, len(test_audio_urls))
print(f"Processing up to {max_samples} samples...")

with torch.no_grad():
    for i, url in enumerate(test_audio_urls[:max_samples]):
        try:
            # Handle local vs remote files
            if url.startswith('http'):
                # Download from URL
                audio_response = session.get(url, headers=headers, timeout=10)
                if audio_response.status_code != 200:
                    print(f"  ⚠ Sample {i}: Failed to download (status {audio_response.status_code})")
                    continue
                audio_bytes = audio_response.content
                audio_stream = io.BytesIO(audio_bytes)
            else:
                # Local file
                audio_stream = url
            
            # Load audio with librosa
            y, sr = librosa.load(audio_stream, sr=16000, duration=10)  # Limit to 10 seconds
            
            # Skip if audio is too short
            if len(y) < 1600:  # Less than 0.1 second
                print(f"  ⚠ Sample {i}: Audio too short ({len(y)} samples)")
                continue
            
            # Create mel spectrogram
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, hop_length=512)
            mel_db = librosa.power_to_db(mel, ref=np.max)
            
            # Normalize
            mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
            
            # Pad/truncate to 500 time frames
            target_frames = 500
            if mel_db.shape[1] < target_frames:
                mel_db = np.pad(mel_db, ((0,0), (0, target_frames - mel_db.shape[1])), mode='constant')
            else:
                mel_db = mel_db[:, :target_frames]
            
            # Convert to tensor and get prediction
            mel_tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            mel_tensor = mel_tensor.to(device)
            
            logits = audio_model(mel_tensor)
            probs = torch.softmax(logits, dim=1)
            
            pred = probs.argmax(dim=1).item()
            pred_prob = probs[0, 1].item()
            
            audio_test_preds.append(pred)
            audio_test_preds_proba.append(pred_prob)
            
            # Get true label
            filename = url.split('/')[-1]
            true_label = test_labels.get(filename, 0)
            audio_test_true.append(true_label)
            
            if (i + 1) % 10 == 0:
                print(f" Processed {i+1}/{max_samples} samples")
                
        except Exception as e:
            print(f" Sample {i} error: {str(e)[:80]}")
            continue

# Convert to numpy arrays
audio_test_true = np.array(audio_test_true)
audio_test_preds = np.array(audio_test_preds)
audio_test_preds_proba = np.array(audio_test_preds_proba)

# Calculate metrics if we have samples
if len(audio_test_true) > 0:
    audio_acc = accuracy_score(audio_test_true, audio_test_preds)
    audio_prec = precision_score(audio_test_true, audio_test_preds, zero_division=0)
    audio_rec = recall_score(audio_test_true, audio_test_preds, zero_division=0)
    audio_f1 = f1_score(audio_test_true, audio_test_preds, zero_division=0)
    
    if len(np.unique(audio_test_true)) > 1:
        audio_auc = roc_auc_score(audio_test_true, audio_test_preds_proba)
    else:
        audio_auc = 0.0
        print(" Only one class present in test data - AUC not meaningful")
else:
    audio_acc = audio_prec = audio_rec = audio_f1 = audio_auc = 0.0
    print("\n No audio samples successfully processed!")

print("\n" + "="*70)
print("AUDIO MODEL EVALUATION RESULTS")
print("="*70)
print(f"Total samples processed: {len(audio_test_true)}")
print(f"Class distribution: Healthy={np.sum(audio_test_true==0)}, Distress={np.sum(audio_test_true==1)}")
print(f"\nMetrics:")
print(f"  Accuracy:  {audio_acc:.4f}")
print(f"  Precision: {audio_prec:.4f}")
print(f"  Recall:    {audio_rec:.4f}")
print(f"  F1-Score:  {audio_f1:.4f}")
print(f"  AUC-ROC:   {audio_auc:.4f}")
print("="*70)

# Store results for later use
audio_test_results = {
    'predictions': audio_test_preds,
    'probabilities': audio_test_preds_proba,
    'true_labels': audio_test_true,
    'accuracy': audio_acc,
    'precision': audio_prec,
    'recall': audio_rec,
    'f1': audio_f1,
    'auc': audio_auc,
    'num_samples': len(audio_test_true)
}

⚠ No GitHub token found - rate limits may apply

FETCHING SPRSOUND TEST DATA

Trying: https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/BioCAS2022/test2022_wav
✓ Found 734 audio files

Fetching test labels...
✓ Loaded 50 labels

PROCESSING AUDIO SAMPLES
Processing up to 50 samples...


In [ ]:
# Audio model test evaluation - stream real audio from SPRSound GitHub
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import io
import json

def create_session_with_retries():
    session = requests.Session()
    retry = Retry(connect=3, backoff_factor=0.5)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

# Load GitHub token for rate limiting
try:
    from dotenv import load_dotenv
    load_dotenv()
    headers = {'Authorization': f"token {os.getenv('GITHUB_TOKEN')}"}
except:
    headers = {}

session = create_session_with_retries()

# Get test audio file URLs (limit to 50 samples as placeholder)
print("Fetching SPRSound test audio URLs from GitHub...")
test_api = "https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/BioCAS2022/test2022_wav"
try:
    response = session.get(test_api, headers=headers, timeout=10)
    test_files = response.json()
    test_audio_urls = [f['download_url'] for f in test_files[:50]]  # First 50 files only
    print(f"✓ Found {len(test_audio_urls)} test audio files")
except Exception as e:
    print(f"⚠ GitHub API error: {str(e)}")
    test_audio_urls = []

# Get test labels
print("Fetching test labels...")
label_api_intra = "https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/BioCAS2022/test2022_json/intra_test_json"
label_api_inter = "https://api.github.com/repos/SJTU-YONGFU-RESEARCH-GRP/SPRSound/contents/BioCAS2022/test2022_json/inter_test_json"

test_labels = {}
for api_url in [label_api_intra, label_api_inter]:
    try:
        response = session.get(api_url, headers=headers, timeout=10)
        label_files = response.json()
        for lf in label_files[:25]:  # Limit labels too
            if lf['name'].endswith('.json'):
                label_response = session.get(lf['download_url'], headers=headers, timeout=10)
                label_data = label_response.json()
                test_labels[lf['name'].replace('.json', '.wav')] = label_data.get('diagnosis_label', 0)
    except Exception as e:
        print(f"⚠ Label fetch error: {str(e)}")

# Process audio samples and get predictions
audio_test_preds = []
audio_test_preds_proba = []
audio_test_true = []

print(f"Processing {len(test_audio_urls)} audio samples...")
with torch.no_grad():
    for i, url in enumerate(test_audio_urls[:50]):  # Process max 50 samples
        try:
            # Download and load audio
            audio_response = session.get(url, headers=headers, timeout=10)
            audio_bytes = audio_response.content
            audio_stream = io.BytesIO(audio_bytes)
            
            # Load audio with librosa
            y, sr = librosa.load(audio_stream, sr=16000)
            
            # Create spectrogram
            mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
            mel_db = librosa.power_to_db(mel)
            mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
            
            # Pad/truncate to 500 frames
            if mel_db.shape[1] < 500:
                mel_db = np.pad(mel_db, ((0,0), (0, 500-mel_db.shape[1])), mode='constant')
            else:
                mel_db = mel_db[:, :500]
            
            # Convert to tensor and get prediction
            mel_tensor = torch.tensor(mel_db).unsqueeze(0).unsqueeze(0).float()
            logits = audio_model(mel_tensor)
            probs = torch.softmax(logits, dim=1)
            
            pred = probs.argmax(dim=1).item()
            pred_prob = probs[0, 1].item()
            
            audio_test_preds.append(pred)
            audio_test_preds_proba.append(pred_prob)
            
            # Get true label from filename
            filename = url.split('/')[-1]
            true_label = test_labels.get(filename, 0)
            audio_test_true.append(true_label)
            
            if (i + 1) % 10 == 0:
                print(f"  ✓ Processed {i+1}/{len(test_audio_urls)} samples")
        except Exception as e:
            print(f"  ⚠ Sample {i} error: {str(e)[:50]}")
            continue

audio_test_preds = np.array(audio_test_preds)
audio_test_preds_proba = np.array(audio_test_preds_proba)
audio_test_true = np.array(audio_test_true)

# Audio metrics
audio_acc = accuracy_score(audio_test_true, audio_test_preds) if len(audio_test_true) > 0 else 0
audio_prec = precision_score(audio_test_true, audio_test_preds, zero_division=0) if len(audio_test_true) > 0 else 0
audio_rec = recall_score(audio_test_true, audio_test_preds, zero_division=0) if len(audio_test_true) > 0 else 0
audio_f1 = f1_score(audio_test_true, audio_test_preds, zero_division=0) if len(audio_test_true) > 0 else 0
audio_auc = roc_auc_score(audio_test_true, audio_test_preds_proba) if len(audio_test_true) > 1 else 0

print(f"\nAudio Model (CNN-LSTM): Acc={audio_acc:.4f} | Prec={audio_prec:.4f} | Rec={audio_rec:.4f} | F1={audio_f1:.4f} | AUC={audio_auc:.4f} | Samples={len(audio_test_true)}")

Fetching SPRSound test audio URLs from GitHub...
⚠ GitHub API error: slice(None, 50, None)
Fetching test labels...
⚠ Label fetch error: slice(None, 25, None)
⚠ Label fetch error: slice(None, 25, None)
Processing 0 audio samples...

Audio Model (CNN-LSTM): Acc=0.0000 | Prec=0.0000 | Rec=0.0000 | F1=0.0000 | AUC=0.0000 | Samples=0


In [ ]:
# Evaluate Fusion Model using ACTUAL audio test predictions
# Align audio and clinical test data
if len(audio_test_preds) > 0:
    min_len = min(len(audio_test_preds), len(y_test))
    audio_test_aligned = audio_test_preds_proba[:min_len]
    clinical_test_aligned = clinical_preds_proba[:min_len]
    y_test_aligned = y_test[:min_len]
    
    # Fusion predictions (70% audio, 30% clinical)
    w_audio_eval = 0.7
    w_clinical_eval = 0.3
    
    fusion_preds_proba_real = (w_audio_eval * audio_test_aligned) + (w_clinical_eval * clinical_test_aligned)
    fusion_preds_real = (fusion_preds_proba_real >= 0.5).astype(int)
    
    # Metrics
    fusion_acc_real = accuracy_score(y_test_aligned, fusion_preds_real)
    fusion_prec_real = precision_score(y_test_aligned, fusion_preds_real, zero_division=0)
    fusion_rec_real = recall_score(y_test_aligned, fusion_preds_real, zero_division=0)
    fusion_f1_real = f1_score(y_test_aligned, fusion_preds_real, zero_division=0)
    fusion_auc_real = roc_auc_score(y_test_aligned, fusion_preds_proba_real)
    
    print(f"Fusion Model (70/30): Acc={fusion_acc_real:.4f} | Prec={fusion_prec_real:.4f} | Rec={fusion_rec_real:.4f} | F1={fusion_f1_real:.4f} | AUC={fusion_auc_real:.4f}")
    
    # Update fusion variables for visualization
    fusion_acc = fusion_acc_real
    fusion_prec = fusion_prec_real
    fusion_rec = fusion_rec_real
    fusion_f1 = fusion_f1_real
    fusion_preds = fusion_preds_real
    y_test = y_test_aligned
else:
    print("⚠ No audio samples processed - using previous synthetic predictions")
    # Keep existing fusion model variables

⚠ No audio samples processed - using previous synthetic predictions


In [ ]:
# Diagnostic: Check audio streaming status
print("="*70)
print("AUDIO STREAMING DIAGNOSTIC")
print("="*70)
print(f"Audio test samples processed: {len(audio_test_true)}")
print(f"Audio test predictions: {len(audio_test_preds)}")
print(f"Audio probabilities: {len(audio_test_preds_proba)}")

if len(audio_test_true) > 0:
    print(f"\n✅ Successfully processed {len(audio_test_true)} real audio samples from SPRSound")
    print(f"   Audio Model Performance:")
    print(f"   - Accuracy: {audio_acc:.4f}")
    print(f"   - Precision: {audio_prec:.4f}")
    print(f"   - Recall: {audio_rec:.4f}")
    print(f"   - F1-Score: {audio_f1:.4f}")
    print(f"   - AUC-ROC: {audio_auc:.4f}")
else:
    print("\n⚠️  No audio samples processed")
    print("   Possible causes:")
    print("   - GitHub API rate limiting (default: 60 req/hour without token)")
    print("   - Network connection timeout")
    print("   - Missing GITHUB_TOKEN in .env file")
    print("\n   ✓ To enable higher rate limits:")
    print("   1. Create GitHub Personal Access Token (Settings > Developer Settings > Tokens)")
    print("   2. Add to .env: GITHUB_TOKEN='your_token_here'")
    print("   3. Re-run the audio evaluation cell")
    print("\n   📌 Note: GitHub Classroom tokens work well for this use case!")
print("="*70)

AUDIO STREAMING DIAGNOSTIC
Audio test samples processed: 0
Audio test predictions: 0
Audio probabilities: 0

⚠️  No audio samples processed
   Possible causes:
   - GitHub API rate limiting (default: 60 req/hour without token)
   - Network connection timeout
   - Missing GITHUB_TOKEN in .env file

   ✓ To enable higher rate limits:
   1. Create GitHub Personal Access Token (Settings > Developer Settings > Tokens)
   2. Add to .env: GITHUB_TOKEN='your_token_here'
   3. Re-run the audio evaluation cell

   📌 Note: GitHub Classroom tokens work well for this use case!


In [ ]:
# Model Comparison - align all to same sample size
if len(audio_test_true) > 0:
    min_samples = min(len(y_test), len(audio_test_true))
    clinical_preds_aligned = clinical_preds[:min_samples]
    y_test_aligned_compare = y_test[:min_samples]
else:
    # If no audio samples, use original clinical test data
    min_samples = len(y_test)
    clinical_preds_aligned = clinical_preds[:min_samples]
    y_test_aligned_compare = y_test[:min_samples]

comparison_df = pd.DataFrame({
    'Model': ['Clinical (ELM)', 'Audio (CNN-LSTM)', 'Fusion (70/30)*'],
    'Accuracy': [clinical_acc, audio_acc, fusion_acc],
    'Precision': [clinical_prec, audio_prec, fusion_prec],
    'Recall': [clinical_rec, audio_rec, fusion_rec],
    'F1-Score': [clinical_f1, audio_f1, fusion_f1],
})

print("\n" + comparison_df.to_string(index=False))
print("\n* Fusion: 70% audio + 30% clinical predictions")
print(f"* Audio evaluation: {'Real SPRSound samples' if len(audio_test_true) > 0 else 'Waiting for GitHub token'}")

# Compute confusion matrices (aligned sizes) - with fallback
if len(audio_test_true) > 0:
    cm_clinical = confusion_matrix(y_test_aligned_compare, clinical_preds_aligned)
    cm_audio = confusion_matrix(audio_test_true, audio_test_preds)
    cm_fusion = confusion_matrix(y_test_aligned_compare, fusion_preds)
    n_cols = 3
else:
    # Without audio samples, show only clinical and fusion
    cm_clinical = confusion_matrix(y_test_aligned_compare, clinical_preds_aligned)
    cm_fusion = confusion_matrix(y_test_aligned_compare, fusion_preds)
    cm_audio = None
    n_cols = 2

# === VISUAL CONFUSION MATRICES ===
if len(audio_test_true) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Clinical Model
    sns.heatmap(cm_clinical, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
                cbar_kws={'label': 'Count'}, 
                xticklabels=['Healthy', 'Distress'], 
                yticklabels=['Healthy', 'Distress'],
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[0].set_title(f'Clinical Model (ELM)\nAcc={clinical_acc:.4f}', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('True Label', fontsize=11)
    axes[0].set_xlabel('Predicted Label', fontsize=11)
    
    # Audio Model
    sns.heatmap(cm_audio, annot=True, fmt='d', cmap='Greens', ax=axes[1],
                cbar_kws={'label': 'Count'},
                xticklabels=['Healthy', 'Distress'],
                yticklabels=['Healthy', 'Distress'],
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[1].set_title(f'Audio Model (CNN-LSTM)\nAcc={audio_acc:.4f}', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('True Label', fontsize=11)
    axes[1].set_xlabel('Predicted Label', fontsize=11)
    
    # Fusion Model
    sns.heatmap(cm_fusion, annot=True, fmt='d', cmap='Oranges', ax=axes[2],
                cbar_kws={'label': 'Count'},
                xticklabels=['Healthy', 'Distress'],
                yticklabels=['Healthy', 'Distress'],
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[2].set_title(f'Fusion Model (70/30)\nAcc={fusion_acc:.4f}', fontsize=13, fontweight='bold')
    axes[2].set_ylabel('True Label', fontsize=11)
    axes[2].set_xlabel('Predicted Label', fontsize=11)
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Clinical Model
    sns.heatmap(cm_clinical, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
                cbar_kws={'label': 'Count'}, 
                xticklabels=['Healthy', 'Distress'], 
                yticklabels=['Healthy', 'Distress'],
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[0].set_title(f'Clinical Model (ELM)\nAcc={clinical_acc:.4f}', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('True Label', fontsize=11)
    axes[0].set_xlabel('Predicted Label', fontsize=11)
    
    # Fusion Model
    sns.heatmap(cm_fusion, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
                cbar_kws={'label': 'Count'},
                xticklabels=['Healthy', 'Distress'],
                yticklabels=['Healthy', 'Distress'],
                annot_kws={'size': 14, 'weight': 'bold'})
    axes[1].set_title(f'Fusion Model (70/30)\nAcc={fusion_acc:.4f}', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('True Label', fontsize=11)
    axes[1].set_xlabel('Predicted Label', fontsize=11)

plt.tight_layout()
plt.show()

if len(audio_test_true) > 0:
    print("\n All three confusion matrices visualized!")
else:
    print("\n Clinical and Fusion confusion matrices visualized!")
    print(" Add GitHub token to .env to enable real audio streaming for audio model evaluation")

NameError: name 'fusion_acc' is not defined

In [ ]:
# === VERY LARGE VISUALIZATION - DETAILED COMPARISON ===
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Confusion matrices (large)
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm_clinical, annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=True,
            xticklabels=['Healthy', 'Distress'], yticklabels=['Healthy', 'Distress'],
            annot_kws={'size': 16, 'weight': 'bold'})
ax1.set_title(f'Clinical (ELM)\nAcc={clinical_acc:.4f} | Prec={clinical_prec:.4f} | Rec={clinical_rec:.4f}', 
              fontsize=12, fontweight='bold')
ax1.set_ylabel('True Label', fontsize=11)
ax1.set_xlabel('Predicted Label', fontsize=11)

ax2 = fig.add_subplot(gs[0, 1])
sns.heatmap(cm_audio, annot=True, fmt='d', cmap='Greens', ax=ax2, cbar=True,
            xticklabels=['Healthy', 'Distress'], yticklabels=['Healthy', 'Distress'],
            annot_kws={'size': 16, 'weight': 'bold'})
ax2.set_title(f'Audio (CNN-LSTM)\nAcc={audio_acc:.4f} | Prec={audio_prec:.4f} | Rec={audio_rec:.4f}', 
              fontsize=12, fontweight='bold')
ax2.set_ylabel('True Label', fontsize=11)
ax2.set_xlabel('Predicted Label', fontsize=11)

ax3 = fig.add_subplot(gs[0, 2])
sns.heatmap(cm_fusion, annot=True, fmt='d', cmap='Oranges', ax=ax3, cbar=True,
            xticklabels=['Healthy', 'Distress'], yticklabels=['Healthy', 'Distress'],
            annot_kws={'size': 16, 'weight': 'bold'})
ax3.set_title(f'Fusion (70/30 audio/clinical)\nAcc={fusion_acc:.4f} | Prec={fusion_prec:.4f} | Rec={fusion_rec:.4f}', 
              fontsize=12, fontweight='bold')
ax3.set_ylabel('True Label', fontsize=11)
ax3.set_xlabel('Predicted Label', fontsize=11)

# Metrics bars
ax4 = fig.add_subplot(gs[1, :])
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
clinical_vals = [clinical_acc, clinical_prec, clinical_rec, clinical_f1]
audio_vals = [audio_acc, audio_prec, audio_rec, audio_f1]
fusion_vals = [fusion_acc, fusion_prec, fusion_rec, fusion_f1]

x_pos = np.arange(len(metrics_names))
width = 0.25

ax4.bar(x_pos - width, clinical_vals, width, label='Clinical (ELM)', color='steelblue', alpha=0.8)
ax4.bar(x_pos, audio_vals, width, label='Audio (CNN-LSTM)', color='seagreen', alpha=0.8)
ax4.bar(x_pos + width, fusion_vals, width, label='Fusion (70/30)', color='orange', alpha=0.8)

ax4.set_ylabel('Score', fontsize=12, fontweight='bold')
ax4.set_title('Performance Metrics Comparison', fontsize=13, fontweight='bold')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(metrics_names, fontsize=11)
ax4.legend(fontsize=11, loc='lower left')
ax4.set_ylim([0, 1.1])
ax4.grid(axis='y', alpha=0.3, linestyle='--')
ax4.axhline(y=0.5, color='red', linestyle='--', alpha=0.3, label='Threshold')

# Sensitivity vs Specificity
ax5 = fig.add_subplot(gs[2, 0])
sensitivity_clinical = cm_clinical[1,1] / (cm_clinical[1,1] + cm_clinical[1,0])
specificity_clinical = cm_clinical[0,0] / (cm_clinical[0,0] + cm_clinical[0,1])

sensitivity_audio = cm_audio[1,1] / (cm_audio[1,1] + cm_audio[1,0])
specificity_audio = cm_audio[0,0] / (cm_audio[0,0] + cm_audio[0,1])

sensitivity_fusion = cm_fusion[1,1] / (cm_fusion[1,1] + cm_fusion[1,0])
specificity_fusion = cm_fusion[0,0] / (cm_fusion[0,0] + cm_fusion[0,1])

models_list = ['Clinical', 'Audio', 'Fusion']
sens_list = [sensitivity_clinical, sensitivity_audio, sensitivity_fusion]
spec_list = [specificity_clinical, specificity_audio, specificity_fusion]

x_models = np.arange(len(models_list))
ax5.bar(x_models - 0.2, sens_list, 0.4, label='Sensitivity (Recall)', color='coral', alpha=0.8)
ax5.bar(x_models + 0.2, spec_list, 0.4, label='Specificity', color='lightblue', alpha=0.8)
ax5.set_ylabel('Score', fontsize=11, fontweight='bold')
ax5.set_title('Sensitivity vs Specificity', fontsize=12, fontweight='bold')
ax5.set_xticks(x_models)
ax5.set_xticklabels(models_list)
ax5.legend(fontsize=10)
ax5.set_ylim([0, 1.1])
ax5.grid(axis='y', alpha=0.3)

# True Positive Rate
ax6 = fig.add_subplot(gs[2, 1])
tpr_clinical = sensitivity_clinical
tpr_audio = sensitivity_audio
tpr_fusion = sensitivity_fusion
fpr_clinical_val = cm_clinical[0,1] / (cm_clinical[0,1] + cm_clinical[0,0])
fpr_audio_val = cm_audio[0,1] / (cm_audio[0,1] + cm_audio[0,0])
fpr_fusion_val = cm_fusion[0,1] / (cm_fusion[0,1] + cm_fusion[0,0])

ax6.scatter([fpr_clinical_val], [tpr_clinical], s=300, label='Clinical', color='steelblue', marker='o', edgecolors='black', linewidth=2)
ax6.scatter([fpr_audio_val], [tpr_audio], s=300, label='Audio', color='seagreen', marker='s', edgecolors='black', linewidth=2)
ax6.scatter([fpr_fusion_val], [tpr_fusion], s=300, label='Fusion', color='orange', marker='^', edgecolors='black', linewidth=2)
ax6.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=2, label='Random')
ax6.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
ax6.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax6.set_title('ROC Space', fontsize=12, fontweight='bold')
ax6.set_xlim([-0.05, 1.05])
ax6.set_ylim([-0.05, 1.05])
ax6.legend(fontsize=10)
ax6.grid(alpha=0.3)

# Model Summary Table
ax7 = fig.add_subplot(gs[2, 2])
ax7.axis('tight')
ax7.axis('off')
summary_data = [
    ['Metric', 'Clinical', 'Audio', 'Fusion'],
    ['Accuracy', f'{clinical_acc:.3f}', f'{audio_acc:.3f}', f'{fusion_acc:.3f}'],
    ['Precision', f'{clinical_prec:.3f}', f'{audio_prec:.3f}', f'{fusion_prec:.3f}'],
    ['Recall', f'{clinical_rec:.3f}', f'{audio_rec:.3f}', f'{fusion_rec:.3f}'],
    ['F1-Score', f'{clinical_f1:.3f}', f'{audio_f1:.3f}', f'{fusion_f1:.3f}'],
]
table = ax7.table(cellText=summary_data, cellLoc='center', loc='center', 
                  colWidths=[0.25, 0.25, 0.25, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)
for i in range(len(summary_data)):
    for j in range(len(summary_data[0])):
        cell = table[(i, j)]
        if i == 0:
            cell.set_facecolor('#40466e')
            cell.set_text_props(weight='bold', color='white')
        else:
            if j == 0:
                cell.set_facecolor('#e8e8e8')
            else:
                cell.set_facecolor('#f0f0f0')

plt.suptitle('FUSION MODEL EVALUATION - ALL MODELS COMPARISON', 
             fontsize=16, fontweight='bold', y=0.995)
plt.show()

print("✅ Comprehensive evaluation complete!")

In [ ]:
# Save Fusion Model Configuration
import json

# Save audio model with torch
torch.jit.save(audio_model, 'cnn_lstm_audio_model_scripted.pt')
print("✓ Audio model saved to 'cnn_lstm_audio_model_scripted.pt'")

# Create fusion model configuration (for reference and inference)
fusion_model_config = {
    'type': 'fusion_model',
    'audio_weight': 0.7,
    'clinical_weight': 0.3,
    'audio_model_path': 'cnn_lstm_audio_model_scripted.pt',
    'clinical_model_path': 'elm_model.pkl',
    'performance': {
        'clinical_acc': float(clinical_acc),
        'audio_acc': float(audio_acc),
        'fusion_acc': float(fusion_acc),
        'clinical_auc': float(clinical_auc),
        'audio_auc': float(audio_auc),
        'fusion_auc': float(fusion_auc),
        'clinical_prec': float(clinical_prec),
        'audio_prec': float(audio_prec),
        'fusion_prec': float(fusion_prec),
        'clinical_rec': float(clinical_rec),
        'audio_rec': float(audio_rec),
        'fusion_rec': float(fusion_rec),
        'clinical_f1': float(clinical_f1),
        'audio_f1': float(audio_f1),
        'fusion_f1': float(fusion_f1),
    }
}

# Save configuration as JSON
with open('fusion_model_config.json', 'w') as f:
    json.dump(fusion_model_config, f, indent=2)

print("✅ Fusion model saved successfully!")
print(f"\n   Configuration:")
print(f"   - Audio weight: {fusion_model_config['audio_weight']}")
print(f"   - Clinical weight: {fusion_model_config['clinical_weight']}")
print(f"\n   Performance:")
print(f"   - Fusion Accuracy: {fusion_acc:.4f}")
print(f"   - Fusion Precision: {fusion_prec:.4f}")
print(f"   - Fusion Recall: {fusion_rec:.4f}")
print(f"   - Fusion F1-Score: {fusion_f1:.4f}")
print(f"   - Fusion AUC-ROC: {fusion_auc:.4f}")
print(f"\n   Files saved:")
print(f"   - fusion_model_config.json (configuration + metrics)")
print(f"   - cnn_lstm_audio_model_scripted.pt (audio model)")
print(f"   - elm_model.pkl (clinical model)")